# 07 -- Robustness (DSR, PBO, reality check)

The anti-overfitting battery. We build a small matrix of parameter variants and ask whether the best one is real or lucky.

In [ ]:
# Parameters (papermill-overridable: `papermill ... -p SYMBOLS '["AAPL","MSFT"]'`)
SYMBOLS = ["ALPHA", "BRAVO", "CHARLIE"]
START = "2018-01-01"
N_DAYS = 600
SEED = 7
USE_SYNTHETIC = True  # set False to fetch real data via core_trading.data.sources


In [ ]:
import numpy as np
import pandas as pd

from core_trading.research.reproducibility import set_seeds

set_seeds(SEED)


def synthetic_bars(symbols, n, seed, start=START):
    """Seeded OHLCV frame in the canonical (symbol, timestamp) layout."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=n, freq="B", tz="UTC")
    frames = []
    for k, sym in enumerate(symbols):
        drift = 0.0003 * (1 + k)
        px = 100.0 + np.cumsum(rng.standard_normal(n) + drift)
        px = np.maximum(px, 1.0)
        high = px + np.abs(rng.standard_normal(n)) * 0.4
        low = px - np.abs(rng.standard_normal(n)) * 0.4
        frame = pd.DataFrame(
            {
                "open": px,
                "high": np.maximum(high, px),
                "low": np.minimum(low, px),
                "close": px,
                "volume": rng.uniform(1e6, 5e6, n),
                "source": "synthetic",
            },
            index=pd.MultiIndex.from_product(
                [[sym], idx], names=["symbol", "timestamp"]
            ),
        )
        frames.append(frame)
    return pd.concat(frames).sort_index()


if USE_SYNTHETIC:
    bars = synthetic_bars(SYMBOLS, N_DAYS, SEED)
else:  # pragma: no cover - exercised only against live vendors
    import asyncio

    from core_trading.data.bars import BarRequest, BarResolution
    from core_trading.data.sources.yfinance_source import YFinanceBarSource

    req = BarRequest(
        symbols=tuple(SYMBOLS),
        resolution=BarResolution.DAY_1,
        start=pd.Timestamp(START, tz="UTC").to_pydatetime(),
        end=pd.Timestamp.now(tz="UTC").to_pydatetime(),
    )
    bars = asyncio.run(YFinanceBarSource().fetch_bars(req))

print(f"loaded {bars.shape[0]} bars across {len(SYMBOLS)} symbols")
bars.head()


In [ ]:
from core_trading.research.feature_store import default_feature_store
from core_trading.research.overfitting import (
    deflated_sharpe_ratio, probability_of_backtest_overfitting,
    whites_reality_check, sharpe_ratio)

store = default_feature_store()
rets = bars['close'].unstack('symbol').pct_change()

# Variants: z-score thresholds for the same mean-reversion signal.
variant_returns = {}
z = store.compute(bars, ['zscore_20'])['zscore_20'].unstack('symbol')
for thr in (0.5, 1.0, 1.5, 2.0, 2.5):
    w = (-z).clip(-thr, thr) / thr
    g = w.abs().sum(axis=1).replace(0.0, np.nan)
    w = w.div(g, axis=0).fillna(0.0)
    r = (w.shift(1) * rets).sum(axis=1).fillna(0.0)
    variant_returns[f'thr_{thr}'] = r
ret_mat = pd.DataFrame(variant_returns).dropna()
ret_mat.apply(sharpe_ratio).round(3)

In [ ]:
best = ret_mat.apply(sharpe_ratio).idxmax()
dsr = deflated_sharpe_ratio(
    ret_mat[best], n_trials=ret_mat.shape[1],
    trial_sharpes=[sharpe_ratio(ret_mat[c], annualised=False)
                   for c in ret_mat.columns])
print(f'best variant: {best}')
print(f'deflated Sharpe: {dsr.deflated_sharpe:.3f} '
      f'(significant={dsr.is_significant})')

In [ ]:
pbo = probability_of_backtest_overfitting(ret_mat.values, n_splits=8)
rc = whites_reality_check(ret_mat.values, n_bootstrap=500, seed=SEED)
print(f'PBO: {pbo.pbo:.3f} (overfit={pbo.is_overfit})')
print(f"White's Reality Check p-value: {rc.pvalue:.3f}")

**Gate:** Deflated Sharpe >= 0.95 **and** PBO < 0.5 **and** reality check significant -> proceed to `08_promotion.ipynb`. Otherwise reject and document why.